## Data structure
Playsession, Snack, Bookreading:
- Videos are `.mov` files
- Audios are `.wav` files

Interview, Language Exposure, Language Experience
- Audios are `.wav` files

## Paths
- Audio files: `/General/Studies/DINA/6_Audio Recordings/WAV Audio Files/`
- Ready for Forced Alignment: `/General/Studies/DINA/Coding/Transcripts/Cleaning Transcripts/{Task}/CLEAN_Ready for Alignment`

### Forced Alignment on snack data

In [ ]:
import os

# This directory contains all the cleaned snack .textgrid transcripts as of 6/5 from the server, as well as the associating .wav file.
working_directory =  'input/DINA_Snack_6_5'

print("All files in this directory:")
print(os.listdir(working_directory))

All files in this directory:
['DINA71_Snack.wav', 'DINA66_Snack.wav', 'DINA71_Clean_Snack.TextGrid', 'DINA76_Snack.wav', 'DINA12_Clean_Snack.TextGrid', 'DINA77_Snack.wav', 'DINA11_Clean_Snack.TextGrid', 'DINA58_Snack.wav', 'DINA76_Clean_Snack.TextGrid', 'DINA80_Snack.wav', 'DINA74_Clean_Snack.TextGrid', 'DINA57_Clean_Snack.TextGrid', 'DINA4_Snack.wav', 'DINA6_Clean_Snack.TextGrid', 'DINA18_Clean_Snack.TextGrid', 'DINA5_Snack.wav', 'DINA77_Clean_Snack.TextGrid', 'DINA5_Clean_Snack.TextGrid', 'DINA7_Clean_Snack.TextGrid', 'DINA74_Snack.wav', 'DINA81_Snack.wav', 'DINA81_Clean_Snack.TextGrid', 'DINA57_Snack.wav', 'DINA48_Clean_Snack.TextGrid', 'DINA37_Snack.wav', 'DINA69_Snack.wav', 'DINA6_Snack.wav', 'DINA80_Clean_Snack.TextGrid', 'DINA9_Clean_Snack.TextGrid', 'DINA9_Snack.wav', 'DINA7_Snack.wav', 'DINA69_Clean_Snack.TextGrid', 'DINA12_Snack.wav', 'DINA58_Clean_Snack.TextGrid', 'DINA4_Clean_Snack.TextGrid', 'DINA11_Snack.wav', 'DINA48_Snack.wav', 'DINA66_Clean_Snack.TextGrid', 'DINA37_Cle

At this point I opened up the TextGrids to visually inspect which one is the *content-corrected* tier. In this case it's tier 3 (index 2 when we're working in 0-index)

In [20]:
import EditPipeline as EP
import AlignPipeline as AP
from lingua import Language

# Here I iterate through every file in this directory.
results = {}
for path in os.listdir(working_directory):

    # Grab participant information
    full_path = os.path.join(working_directory, path)
    participant_id = path.split("_")[0].split("DINA")[1]

    # For each textgrid, I do the forced alignment
    if ".textgrid" in str(full_path).lower():
        
        # Grab the associating .wav file
        audio_path = os.path.join(working_directory, f"DINA{participant_id}_Snack.wav")
        
        # Extract the json of the 3rd tier
        segments = EP.textgrid_to_json(tg_path=full_path, tier_index=2)

        # Preprocessing: MFA can only take transcript utterances that exist in the audio.
        # So things like "sil" or "[3rd Party]" are undetectable from MFA
        print("Before processing:", segments)
        segments_cleaned = []
        for segment in segments:
            include = True
            if segment['text'][0] == '[' and segment['text'][-1] == ']': # brackets, remove
                include = False
            if 'sil' in segment['text'].lower():
                include = False
            if include: 
                segments_cleaned.append(segment)
        for segment in segments:
            if segment not in segments_cleaned:
                print("Removed:", segment)
        print("After processing:", segments_cleaned)

        # Run the forced alignment script
        try:
            aligned_segments = AP.script(audio_path=audio_path, transcript=segments_cleaned, languages=[Language.ENGLISH, Language.SPANISH])
        except Exception as e:
            aligned_segments = None
            print("Couldn't do forced alignment for participant:", participant_id, full_path, audio_path)

        # Append
        results[participant_id] = aligned_segments

Before processing: [{'start': 0.0044, 'end': 9.899, 'text': "We've never tried a pouch before. We've never tried it. I don't know. I don't know. I feel like you would try it. Yeah, you're like, I don't know what to do with this."}, {'start': 9.899, 'end': 12.9252, 'text': 'sil'}, {'start': 12.9252, 'end': 15.051, 'text': '[Laugh]'}, {'start': 15.051, 'end': 18.28, 'text': 'sil'}, {'start': 18.28, 'end': 19.34, 'text': 'Oh!'}, {'start': 19.34, 'end': 19.85, 'text': 'sil'}, {'start': 19.85, 'end': 20.37, 'text': '[Gasp]'}, {'start': 20.37, 'end': 21.309, 'text': 'sil'}, {'start': 21.309, 'end': 23.09, 'text': 'Yeah.'}, {'start': 23.09, 'end': 27.059, 'text': 'sil'}, {'start': 27.059, 'end': 29.469, 'text': 'Mm, is that tasty?'}, {'start': 29.469, 'end': 31.12, 'text': 'sil'}, {'start': 31.12, 'end': 33.65, 'text': 'Oh, good job.'}, {'start': 33.65, 'end': 47.279, 'text': 'sil'}, {'start': 47.279, 'end': 49.9, 'text': 'Good job, yeah. [Gasp]'}, {'start': 49.9, 'end': 51.509, 'text': 'sil'